# Sri Sivasubramaniya Nadar College of Engineering, Chennai
### (An Autonomous Institution Affiliated to Anna University)
**Degree & Branch:** M. Tech (Integrated) Computer Science & Engineering | **Semester:** V
**Subject Code & Name:** ICS1512 & Machine Learning Algorithms Laboratory
**Academic Year:** 2025-2026 (Odd) | **Batch:** 2024-2029

---
## Experiment 9: Perceptron vs Multilayer Perceptron (A/B Experiment) with Hyperparameter Tuning

**Student Name:** Danusu K | **Register Number:** 3122247001013
**Faculty:** Dr. Poreddy Ajay Kumar Reddy

### Objectives:
1. Implement **Model A: Single-Layer Perceptron Learning Algorithm (PLA)** from scratch (step activation, classical weight-update rule) as a multiclass One-vs-Rest classifier.
2. Implement **Model B: Multilayer Perceptron (MLP)** with hidden layers and non-linear activations, trained via backpropagation.
3. Select and justify MLP hyperparameters (activation, cost function, optimizer, learning rate, architecture, batch size) through systematic staged hyperparameter tuning.
4. Compare PLA and tuned MLP using Accuracy, Precision, Recall, F1-score, confusion matrices, micro/macro-average ROC curves, and training convergence curves on the **English Handwritten Characters** dataset (62 classes).

## 1. Environment Setup & Library Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_curve, auc
)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment initialized successfully with seed = 42.")

## 2. Dataset Ingestion & Preprocessing
Load the English Handwritten Characters dataset (3,410 images, 62 balanced classes), convert to grayscale, resize to 28x28, flatten to 784-d, and normalize to [0, 1].

In [ ]:
DATA_DIR = "../dataset"
IMG_SIZE = 28

df = pd.read_csv(os.path.join(DATA_DIR, "english.csv"))
X = np.zeros((len(df), IMG_SIZE * IMG_SIZE), dtype=np.float32)
for i, rel_path in enumerate(df["image"]):
    img = Image.open(os.path.join(DATA_DIR, rel_path)).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    X[i] = np.asarray(img, dtype=np.float32).flatten() / 255.0

le = LabelEncoder()
y = le.fit_transform(df["label"].astype(str))
class_names = le.classes_
n_classes = len(class_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Loaded {X.shape[0]} images | Feature dim: {X.shape[1]} | Classes: {n_classes}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 3. Model A — Perceptron Learning Algorithm (PLA), From Scratch
Multiclass One-vs-Rest Perceptron: step activation, classical update rule $\mathbf{w}_{t+1} = \mathbf{w}_t + \eta(y-\hat{y})\mathbf{x}$, applied simultaneously (vectorized) across all 62 sub-classifiers for each training sample, each epoch.

In [ ]:
def run_perceptron_ovr(X_train, y_train, X_test, y_test, n_classes, epochs=60, eta=0.1, seed=RANDOM_STATE):
    n_train, n_features = X_train.shape
    Xb_train = np.hstack([X_train, np.ones((n_train, 1))])
    Xb_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
    Y_onehot = np.eye(n_classes)[y_train]

    W = np.zeros((n_classes, n_features + 1))
    rng = np.random.RandomState(seed)
    history = []
    for epoch in range(1, epochs + 1):
        for idx in rng.permutation(n_train):
            x = Xb_train[idx]
            y_hat = (W @ x >= 0).astype(np.float64)
            error = Y_onehot[idx] - y_hat
            if np.any(error != 0):
                W += eta * np.outer(error, x)
        train_acc = np.mean(np.argmax(Xb_train @ W.T, axis=1) == y_train)
        test_acc = np.mean(np.argmax(Xb_test @ W.T, axis=1) == y_test)
        history.append({"epoch": epoch, "train_accuracy": train_acc, "test_accuracy": test_acc})
    test_scores = Xb_test @ W.T
    return W, history, np.argmax(test_scores, axis=1), test_scores

W_pla, pla_history, pla_pred, pla_scores = run_perceptron_ovr(X_train, y_train, X_test, y_test, n_classes, epochs=60)
print(f"PLA final test accuracy: {pla_history[-1][\'test_accuracy\']*100:.2f}%")

## 4. Model B — Multilayer Perceptron (MLP): Staged Hyperparameter Tuning
Stage 1 tunes architecture x activation; Stage 2 tunes optimizer x learning rate (using the Stage 1 winner); Stage 3 tunes batch size (using the Stage 1+2 winner). Each stage uses 3-fold Stratified Cross-Validation.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_grid_1 = {"hidden_layer_sizes": [(64,), (128,), (128, 64), (256, 128, 64)], "activation": ["relu", "tanh", "logistic"]}
gs1 = GridSearchCV(
    MLPClassifier(solver="adam", learning_rate_init=0.001, batch_size=32, max_iter=150, random_state=RANDOM_STATE),
    param_grid_1, cv=cv, scoring="accuracy", n_jobs=-1
)
gs1.fit(X_train, y_train)
print("Stage 1 best:", gs1.best_params_, "CV Acc:", f"{gs1.best_score_*100:.2f}%")

In [ ]:
best_arch = gs1.best_params_["hidden_layer_sizes"]
best_activation = gs1.best_params_["activation"]

param_grid_2 = {"solver": ["sgd", "adam"], "learning_rate_init": [0.0001, 0.001, 0.01, 0.1]}
gs2 = GridSearchCV(
    MLPClassifier(hidden_layer_sizes=best_arch, activation=best_activation, batch_size=32, max_iter=150, random_state=RANDOM_STATE),
    param_grid_2, cv=cv, scoring="accuracy", n_jobs=-1
)
gs2.fit(X_train, y_train)
print("Stage 2 best:", gs2.best_params_, "CV Acc:", f"{gs2.best_score_*100:.2f}%")

In [ ]:
best_solver = gs2.best_params_["solver"]
best_lr = gs2.best_params_["learning_rate_init"]

param_grid_3 = {"batch_size": [16, 32, 64, 128]}
gs3 = GridSearchCV(
    MLPClassifier(hidden_layer_sizes=best_arch, activation=best_activation, solver=best_solver,
                  learning_rate_init=best_lr, max_iter=150, random_state=RANDOM_STATE),
    param_grid_3, cv=cv, scoring="accuracy", n_jobs=-1
)
gs3.fit(X_train, y_train)
best_batch_size = gs3.best_params_["batch_size"]
print("Stage 3 best batch_size:", best_batch_size, "CV Acc:", f"{gs3.best_score_*100:.2f}%")

best_config = {"hidden_layer_sizes": best_arch, "activation": best_activation,
               "solver": best_solver, "learning_rate_init": best_lr, "batch_size": best_batch_size}
print("\nFinal chosen MLP configuration:", best_config)

## 5. Consolidated Results (from `experiment9_results.json`)

In [ ]:
with open("experiment9_results.json", "r") as f:
    results = json.load(f)

print("Dataset:", results["dataset_info"])
print("\nPLA test metrics:", results["pla"]["test_metrics"])
print("\nMLP best config:", results["mlp"]["best_config"])
print("\nMLP test metrics:", results["mlp"]["test_metrics"])

In [ ]:
comparison = pd.DataFrame([
    {"Model": "PLA", **results["pla"]["test_metrics"], "ROC_AUC_micro": results["pla"]["roc"]["auc_micro"]},
    {"Model": "MLP (Tuned)", **results["mlp"]["test_metrics"], "ROC_AUC_micro": results["mlp"]["roc"]["auc_micro"]},
])
comparison

## 6. Visualizations
All plots below are generated by `experiment9.py` and saved to `../output_plots/`.

In [ ]:
from IPython.display import Image as IPImage, display

plot_files = [
    "../output_plots/01_sample_character_gallery.png",
    "../output_plots/02_pla_learning_rate_sensitivity.png",
    "../output_plots/03_pla_training_convergence.png",
    "../output_plots/04_mlp_architecture_activation_tuning.png",
    "../output_plots/05_mlp_optimizer_lr_comparison.png",
    "../output_plots/06_mlp_batch_size_comparison.png",
    "../output_plots/07_pla_vs_mlp_convergence_curves.png",
    "../output_plots/08_confusion_matrices_pla_vs_mlp.png",
    "../output_plots/09_roc_curves_micro_macro.png",
    "../output_plots/10_performance_metrics_comparison.png",
    "../output_plots/11_perceptron_weight_visualization.png",
]
for p in plot_files:
    if os.path.exists(p):
        print(f"Displaying: {os.path.basename(p)}")
        display(IPImage(filename=p, width=800))

## 7. Key Conclusions & Insights
1. **PLA never converges.** On this non-linearly-separable 62-class problem, PLA's train/test accuracy and misclassification counts oscillate indefinitely across all 60 epochs rather than stabilizing, finishing at only **18.62%** test accuracy (macro-F1 16.04%).
2. **Learning rate is theoretically irrelevant to PLA's decision boundary.** With zero-initialized weights, test accuracy was *exactly identical* (14.96%) across $\eta \in \{0.001, 0.01, 0.1, 1.0\}$ — confirming that step-activation perceptron updates are scale-invariant.
3. **ReLU catastrophically "dies" on this dataset** (~1.5% accuracy, chance level) while **Logistic (Sigmoid)** activation reaches ~40% CV accuracy at the same architecture — the single largest hyperparameter effect observed.
4. **Deeper MLPs consistently underperformed** a single hidden layer (128,) given only ~44 training images per class — a clear bias-variance trade-off with limited data.
5. **Adam converges faster but is far less forgiving of learning-rate misspecification than SGD**: Adam collapsed to chance level at lr $\ge$ 0.01, while SGD degraded gracefully across the same range.
6. **Final result:** the tuned MLP (single hidden layer, Logistic, Adam, lr=0.001, batch_size=64) more than doubled PLA's test accuracy (18.62% $\to$ 38.56%) and macro-F1 (16.04% $\to$ 37.93%), with ROC-AUC improving from 0.80 to 0.91 (micro-average).